Hypothesis H-B: Subjective sleep quality (SSQ) and objective sleep quality (OSQ), represented individually by total sleep time (TST), sleep onset latency (SOL), wake after sleep onset (WASO), and sleep efficiency (SE), show meaningful correlation across subjects.

The hypothesis will be tested in a longitudinal, multi-subject dataset.

**Please note that the analyses shown in this notebook were performed on synthetic data, and the results do not reflect the actual outcomes of the study.**

For more details, see the related preregistration at osf.io.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
import matplotlib.dates as mdates
import statsmodels.formula.api as smf

In [ ]:
strength = 50  # a slider [0, 100] that determines the amount of randomness in data;
               # 0: more random, less correlation; 100: less random: more correlation
rng = np.random.default_rng(9)  # to keep things reproducible

In [ ]:
n_subjects = 700
days_subject = 21
start_date = "2000-01-03"
end_date   = "2000-01-23"
rng = np.random.default_rng(9)

In [ ]:
def quality_from_likert(likert):
    return (likert - 1) / 4.0

def synthesize_metric(q, low_val, high_val, strength_0_100=0, positive=True, jitter=0.0, rng=None):
    s = np.clip(strength_0_100 / 100.0, 0.0, 1.0)
    base_noise = rng.random(len(q))
    trend = q if positive else (1.0 - q)
    z = (1.0 - s) * base_noise + s * trend
    if jitter > 0:
        z = np.clip(z + rng.normal(0, jitter, size=len(z)), 0, 1)
    return low_val + z * (high_val - low_val)

In [ ]:
# Generating mock data

mondays = pd.date_range(start=start_date, end=end_date, freq="W-MON")
valid_mondays = mondays[mondays + pd.Timedelta(days=days_subject - 1) <= pd.to_datetime(end_date)]

rows = []
for sub_id in range(1, n_subjects + 1):
    sdate = rng.choice(valid_mondays)
    dates = pd.date_range(start=sdate, periods=days_subject, freq="D")

    # week indices 0,1,2 for the three calendar weeks (Mon-Sun)
    week_idx = ((dates - dates[0]).days // 7).astype(int)

    # one Subj_likert per calendar week
    week_likerts = rng.integers(1, 6, size=3)
    subj_likert = week_likerts[week_idx]

    q = quality_from_likert(subj_likert)

    TST  = synthesize_metric(q, 300, 600, strength, positive=True,  jitter=0.01, rng=rng).round().astype(int)
    SE   = np.round(synthesize_metric(q, 0.60, 0.98, strength, positive=True,  jitter=0.01, rng=rng), 3)
    SOL  = synthesize_metric(q, 10, 100, strength, positive=False, jitter=0.01, rng=rng).round().astype(int)
    WASO = synthesize_metric(q, 10, 100, strength, positive=False, jitter=0.01, rng=rng).round().astype(int)

    rows.append(pd.DataFrame({
        "date": dates.strftime("%Y-%m-%d"),
        "Sub_id": sub_id,
        "day_id": np.arange(1, days_subject + 1),
        "TST": TST,
        "SE": SE,
        "SOL": SOL,
        "WASO": WASO,
        "Subj_likert": subj_likert
    }))

dataset_n700 = pd.concat(rows, ignore_index=True)
print("Mock HBS daily data:\n")
print(dataset_n700.iloc[0:25, :].to_string(index=False))

Mock HBS daily data:

      date  Sub_id  day_id  TST    SE  SOL  WASO  Subj_likert
2000-01-03       1       1  473 0.870   66    44            3
2000-01-04       1       2  491 0.837   63    66            3
2000-01-05       1       3  481 0.755   35    40            3
2000-01-06       1       4  515 0.731   68    38            3
2000-01-07       1       5  501 0.810   75    45            3
2000-01-08       1       6  515 0.808   48    70            3
2000-01-09       1       7  375 0.748   64    56            3
2000-01-10       1       8  517 0.816   49    12            5
2000-01-11       1       9  520 0.933   30    18            5
2000-01-12       1      10  454 0.914   17    35            5
2000-01-13       1      11  449 0.943   41    13            5
2000-01-14       1      12  570 0.833   48    23            5
2000-01-15       1      13  599 0.922   25    19            5
2000-01-16       1      14  568 0.804   20    42            5
2000-01-17       1      15  499 0.794   44    17

In [ ]:
# calculating per subject means

subj_means = (
    dataset_n700
    .groupby('Sub_id', as_index=False)
    .agg(TST=('TST','mean'),
         SE=('SE','mean'),
         SOL=('SOL','mean'),
         WASO=('WASO','mean'),
         Subj_likert=('Subj_likert','median')))

print("Mock HBS subj mean data:\n")
print(subj_means.iloc[0:10, :].to_string(index=False))

Mock HBS subj mean data:

 Sub_id        TST       SE       SOL      WASO  Subj_likert
      1 514.095238 0.852476 40.238095 34.238095          5.0
      2 456.142857 0.778571 53.952381 59.142857          2.0
      3 425.095238 0.760524 59.285714 61.619048          2.0
      4 423.333333 0.764714 64.333333 56.190476          2.0
      5 452.380952 0.795381 52.428571 51.000000          4.0
      6 432.238095 0.758286 58.952381 60.809524          3.0
      7 455.190476 0.785333 54.095238 53.190476          3.0
      8 458.238095 0.772524 58.333333 53.476190          2.0
      9 428.000000 0.762810 65.047619 59.000000          3.0
     10 414.142857 0.765810 65.714286 60.190476          2.0


In [ ]:
# Simple moving block bootstrap CI for correlation (serial-aware for daily)

rng_boot = np.random.default_rng(9)

def block_boot_ci(x, y, method="spearman", block=None, reps=5000, alpha=0.05):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    ok = ~np.isnan(x) & ~np.isnan(y)
    x, y = x[ok], y[ok]
    n = len(x)
    if n < 3:
        return (np.nan, np.nan, np.nan)

    if method == "spearman":
        def corr(a, b):
            return spearmanr(a, b).correlation
    else:
        def corr(a, b):
            return pearsonr(a, b)[0]

    # subject-level resampling
    stat_hat = corr(x, y)
    boot = np.empty(reps)
    for b in range(reps):
        idx = rng_boot.integers(0, n, size=n)
        boot[b] = corr(x[idx], y[idx])

    lo, hi = np.percentile(boot, [100*alpha/2, 100*(1-alpha/2)])
    return float(stat_hat), float(lo), float(hi)

In [ ]:
def outcome(df, level_label, threshold=0.20):
    def label_target(t):
        t_str = str(t).lower()
        if "slide" in t_str:  return "SSQ (slider)"
        if "likert" in t_str: return "SSQ (Likert)"
        return str(t)
    methods = [
        ("Spearman", "spearman_rho", "spearman_ci_lo", "spearman_ci_hi", "ρ"),
        ("Pearson",  "pearson_r",    "pearson_ci_lo",   "pearson_ci_hi",  "r")]

    have = {m[0]: set(m[1:4]).issubset(df.columns) for m in methods}

    for _, row in df.iterrows():
        metric = row.get("metric", "metric")
        target = label_target(row.get("target", "target"))
        for name, eff_col, lo_col, hi_col, sym in methods:
            if not have[name]:
                continue
            eff = float(row[eff_col]); lo = float(row[lo_col]); hi = float(row[hi_col])
            ci_excludes_zero = (lo > 0) or (hi < 0)
            meets_thresh = abs(eff) >= threshold
            accepted = ci_excludes_zero and meets_thresh
            print(
                f"{name} correlation between ({level_label}) {metric} and {target} "
                f"was {eff:.3f} (95% CI: {lo:.3f}–{hi:.3f}); "
                f"|{sym}|≥{threshold:.2f} is {'met' if meets_thresh else 'not met'}; "
                f"CI excludes 0 is {'yes' if ci_excludes_zero else 'no'}; "
                f"{'Hypothesis is accepted' if accepted else 'Hypothesis is rejected'}.")

In [ ]:
df = subj_means.copy()
metrics = ["TST", "SE", "SOL", "WASO"]
targets = ["Subj_likert"]

In [ ]:
# calculating Spearman correlations with 95% CI

subj_rows = []
for metric in metrics:
    for target in targets:
        # Spearman ρ with bootstrap 95% CI via subject resampling
        rho_hat, rho_lo, rho_hi = block_boot_ci(df[metric], df[target], method="spearman", block=None, reps=5000)
        # two-sided p-value for FDR (from Spearman test)
        _rho, pval = spearmanr(df[metric], df[target])
        subj_rows.append({
            "metric": metric,
            "target": target,
            "n_obs": int(df[[metric, target]].dropna().shape[0]),
            "spearman_rho": float(rho_hat),
            "spearman_ci_lo": float(rho_lo),
            "spearman_ci_hi": float(rho_hi),
            "spearman_p": float(pval)
        })

subj_results = pd.DataFrame(subj_rows)
subj_results[["spearman_rho","spearman_ci_lo","spearman_ci_hi"]] = subj_results[["spearman_rho","spearman_ci_lo","spearman_ci_hi"]].round(4)

print("\nBetween-subject correlations with 95% CI (bootstrap over subjects):\n")
print(subj_results.to_string(index=False))


Between-subject correlations with 95% CI (bootstrap over subjects):

metric      target  n_obs  spearman_rho  spearman_ci_lo  spearman_ci_hi    spearman_p
   TST Subj_likert    700        0.8552          0.8339          0.8736 2.132652e-201
    SE Subj_likert    700        0.8480          0.8262          0.8677 1.332276e-194
   SOL Subj_likert    700       -0.8581         -0.8756         -0.8382 3.284922e-204
  WASO Subj_likert    700       -0.8578         -0.8761         -0.8360 6.156413e-204


In [ ]:
outcome(subj_results, "aggregated", threshold=0.20)

Spearman correlation between (aggregated) TST and SSQ (Likert) was 0.855 (95% CI: 0.834–0.874); |ρ|≥0.20 is met; CI excludes 0 is yes; Hypothesis is accepted.
Spearman correlation between (aggregated) SE and SSQ (Likert) was 0.848 (95% CI: 0.826–0.868); |ρ|≥0.20 is met; CI excludes 0 is yes; Hypothesis is accepted.
Spearman correlation between (aggregated) SOL and SSQ (Likert) was -0.858 (95% CI: -0.876–-0.838); |ρ|≥0.20 is met; CI excludes 0 is yes; Hypothesis is accepted.
Spearman correlation between (aggregated) WASO and SSQ (Likert) was -0.858 (95% CI: -0.876–-0.836); |ρ|≥0.20 is met; CI excludes 0 is yes; Hypothesis is accepted.


In [ ]:
# simple linear regression (OLS):

df = weekly_df.copy()
df["Sub_id"] = df["Sub_id"].astype(int)
df["Subj_likert"] = pd.to_numeric(df["Subj_likert"], errors="coerce")
metrics = ["TST","SE","SOL","WASO"]

# OLS on subject-level aggregated data; scale predictors for interpretability
subj = subj_means.assign(
    TST_h   = subj_means["TST"]  / 60.0,   # hours
    SE_10p  = subj_means["SE"]   * 10.0,   # 10 percentage-point steps
    SOL_10  = subj_means["SOL"]  / 10.0,   # 10-minute steps
    WASO_10 = subj_means["WASO"] / 10.0    # 10-minute steps
)

ols_map = {
    "TST_h": "TST_h",
    "SE_10p": "SE_10p",
    "SOL_10": "SOL_10",
    "WASO_10": "WASO_10"}

rows = []
for x in ols_map.values():
    res = smf.ols(f"Subj_likert ~ {x}", data=subj).fit()
    ci  = res.conf_int().loc[x]
    rows.append({
        "predictor": x,
        "beta": res.params[x],
        "ci_low": ci[0],
        "ci_high": ci[1],
        "pvalue": res.pvalues[x],
        "n_subjects": int(res.nobs)})

ols_res = pd.DataFrame(rows)
print("\nOLS (scaled predictors) with 95% CIs:\n")
print(ols_res.to_string(index=False))


Between-subject OLS (scaled predictors) with 95% CIs:

predictor      beta    ci_low   ci_high        pvalue  n_subjects
    TST_h  1.795954  1.711941  1.879966 4.427856e-193         700
   SE_10p  2.367323  2.254773  2.479873 1.413955e-189         700
   SOL_10 -1.025238 -1.072297 -0.978180 3.197846e-197         700
  WASO_10 -1.011994 -1.058329 -0.965658 9.118974e-198         700


In [50]:
def print1(row):
  print(f"{row['predictor']}: β ≈ {round(row['beta'], 2)} (95% CI: {round(row['ci_low'], 2)} – {round(row['ci_high'], 2)})")
print1(ols_res.iloc[0])
print(f"\tParticipants who sleep 1 hour more (on average) report {round(ols_res['beta'][0], 2)} points higher SSQ")
print1(ols_res.iloc[1])
print(f"\tA 10-percentage-point higher sleep efficiency is associated with {round(ols_res['beta'][1], 2)} points higher SSQ")
print1(ols_res.iloc[2])
print(f"\tParticipants who took 10 minutes longer to fall asleep report {-round(ols_res['beta'][2], 2)} points lower SSQ")
print1(ols_res.iloc[3])
print(f"\tParticipants who stay awake 10 minutes longer during the night report {-round(ols_res['beta'][3], 2)} points lower SSQ")

TST_h: β ≈ 1.8 (95% CI: 1.71 – 1.88)
	Participants who sleep 1 hour more (on average) report 1.8 points higher SSQ
SE_10p: β ≈ 2.37 (95% CI: 2.25 – 2.48)
	A 10-percentage-point higher sleep efficiency is associated with 2.37 points higher SSQ
SOL_10: β ≈ -1.03 (95% CI: -1.07 – -0.98)
	Participants who took 10 minutes longer to fall asleep report 1.03 points lower SSQ
WASO_10: β ≈ -1.01 (95% CI: -1.06 – -0.97)
	Participants who stay awake 10 minutes longer during the night report 1.01 points lower SSQ


**######################################################**

In [45]:
# Generate entirely random data (no seed)

start_date = "2000-01-03"
end_date = "2003-01-23"
num_subjects = 700
rng = np.random.default_rng(42)

trial_dates = pd.date_range(start="2000-01-01", end="2000-01-03", freq="D").strftime('%Y-%m-%d')
n_trials = len(trial_dates)

# Build repeating arrays: for each subject, the three dates and trial ids 1..3
dates = np.tile(trial_dates, num_subjects)
sub_ids = np.repeat(np.arange(1, num_subjects + 1), n_trials)
trial_ids = np.tile(np.arange(1, n_trials + 1), num_subjects)

# Sanity check lengths
rows = len(dates)
assert rows == num_subjects * n_trials, "Row count mismatch"

# Random columns (same ranges as earlier)
TST = rng.integers(300, 601, size=rows)             # total sleep time in minutes (300-600)
SE = np.round(rng.random(rows), 3)                  # sleep efficiency (0-1)
SOL = rng.integers(10, 101, size=rows)              # sleep onset latency in minutes (10-100)
WASO = rng.integers(10, 101, size=rows)             # wake after sleep onset in minutes (10-100)
Subj = rng.integers(1, 6, size=rows)                # Likert 1-5 (subjective)

hbs = pd.DataFrame({
    "date": dates,
    "Sub_id": sub_ids,
    "week_id": trial_ids,
    "TST": TST,
    "SE": SE,
    "SOL": SOL,
    "WASO": WASO,
    "Subj_likert": Subj})

In [ ]:
# aggregating daily data into weekly bins

daily_df = dataset_n700.copy()
daily_df['date'] = pd.to_datetime(daily_df['date'])
daily_df['week_start'] = daily_df['date'].dt.to_period('W-MON').apply(lambda p: p.start_time)

weekly_df = (
    daily_df
    .groupby(['Sub_id', 'week_start'], sort=True)
    .agg(
        TST=('TST', 'mean'),
        SE =('SE', 'mean'),
        SOL=('SOL', 'mean'),
        WASO=('WASO', 'mean'),
        Subj_likert=('Subj_likert', 'median')
    ).reset_index())

weekly_df['day_id'] = weekly_df.groupby('Sub_id').cumcount() + 1
weekly_df['date'] = weekly_df['week_start'].dt.strftime('%Y-%m-%d')
weekly_df = weekly_df[['date', 'Sub_id', 'day_id', 'TST', 'SE', 'SOL', 'WASO', 'Subj_likert']]
weekly_df.rename(columns={'day_id': 'week_id'}, inplace=True)
print("Mock HBS weekly data:\n")
print(weekly_df.iloc[0:20, :].to_string(index=False))

Mock HBS weekly data:

      date  Sub_id  week_id        TST       SE       SOL      WASO  Subj_likert
1999-12-28       1        1 473.000000 0.870000 66.000000 44.000000          3.0
2000-01-04       1        2 485.000000 0.786429 57.428571 46.714286          3.0
2000-01-11       1        3 522.714286 0.877571 32.142857 23.857143          5.0
2000-01-18       1        4 544.833333 0.897333 25.333333 30.166667          5.0
1999-12-28       2        1 459.000000 0.792000 49.000000 67.000000          2.0
2000-01-04       2        2 415.714286 0.708857 52.571429 70.428571          2.0
2000-01-11       2        3 497.428571 0.852714 46.000000 41.571429          4.0
2000-01-18       2        4 454.666667 0.771167 65.666667 65.166667          2.0
1999-12-28       3        1 369.000000 0.726000 44.000000 47.000000          2.0
2000-01-04       3        2 408.571429 0.746571 64.000000 61.571429          2.0
2000-01-11       3        3 416.000000 0.781714 62.428571 66.285714          2.0
2000-